# MLP for Image Classification (CIFAR-10)

In [1]:
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

Check GPU Availability

In [3]:
# Check if CUDA is available
if torch.cuda.is_available():
    print("CUDA is available!")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"Current device: {torch.cuda.current_device()}")
    print(f"Device name: {torch.cuda.get_device_name(0)}")

    device = torch.device("cuda")
else:
    print("CUDA is not available. Using CPU.")
    device = torch.device("cpu")

CUDA is available!
Number of GPUs: 1
Current device: 0
Device name: NVIDIA GeForce RTX 3050 Ti Laptop GPU


Define Transformations

In [4]:
# Define the transformation applied to RGB data before coming into the network
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

Load CIFAR-10 Dataset

In [5]:
train_data = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    transform=transform,
    download=False
)

test_data = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    transform=transform,
    download=False
)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

c:\Users\72786\anaconda3\envs\multimodal_ai\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Inspect Data

In [6]:
class_names = ['plane', 'car', 'bird', 'cat',
               'deer', 'dog', 'frog', 'horse',
               'ship', 'truck']

image, label = train_data[0]

print(image.size())
print(f"Label index: {label}")
print(f"Class name: {class_names[label]}")

torch.Size([3, 32, 32])
Label index: 6
Class name: frog


Define MLP Model

In [7]:
class MLPNet(nn.Module):

    def __init__(self):
        super().__init__()

        # CIFAR-10 image size: 3 channels x 32 pixels x 32 pixels = 3072 features
        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(3 * 32 * 32, 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, 256)
        self.fc4 = nn.Linear(256, 10)

        self.dropout = nn.Dropout(0.3)

    def forward(self, x):

        # Flatten image into vector
        x = self.flatten(x)

        x = F.relu(self.fc1(x))
        x = self.dropout(x)

        x = F.relu(self.fc2(x))
        x = self.dropout(x)

        x = F.relu(self.fc3(x))

        x = self.fc4(x)

        return x

Initialize Model, Loss Function, and Optimizer

In [8]:
net = MLPNet().to(device)

loss_function = nn.CrossEntropyLoss()

optimizer = optim.SGD(
    net.parameters(),
    lr=0.001,
    momentum=0.9
)

Train the Model

In [9]:
for epoch in range(30):

    print(f'Training epoch {epoch + 1} ...')

    running_loss = 0.0

    net.train()

    for data in train_loader:

        inputs, labels = data

        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = net(inputs)

        loss = loss_function(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(f'Loss: {running_loss / len(train_loader):.4f}')

Training epoch 1 ...
Loss: 2.2152
Training epoch 2 ...
Loss: 1.9408
Training epoch 3 ...
Loss: 1.7806
Training epoch 4 ...
Loss: 1.6837
Training epoch 5 ...
Loss: 1.6163
Training epoch 6 ...
Loss: 1.5619
Training epoch 7 ...
Loss: 1.5118
Training epoch 8 ...
Loss: 1.4678
Training epoch 9 ...
Loss: 1.4269
Training epoch 10 ...
Loss: 1.3889
Training epoch 11 ...
Loss: 1.3573
Training epoch 12 ...
Loss: 1.3244
Training epoch 13 ...
Loss: 1.2953
Training epoch 14 ...
Loss: 1.2696
Training epoch 15 ...
Loss: 1.2389
Training epoch 16 ...
Loss: 1.2144
Training epoch 17 ...
Loss: 1.1939
Training epoch 18 ...
Loss: 1.1663
Training epoch 19 ...
Loss: 1.1488
Training epoch 20 ...
Loss: 1.1225
Training epoch 21 ...
Loss: 1.0993
Training epoch 22 ...
Loss: 1.0784
Training epoch 23 ...
Loss: 1.0595
Training epoch 24 ...
Loss: 1.0408
Training epoch 25 ...
Loss: 1.0180
Training epoch 26 ...
Loss: 1.0002
Training epoch 27 ...
Loss: 0.9785
Training epoch 28 ...
Loss: 0.9601
Training epoch 29 ...
Loss: 0

Save the Trained Model

In [10]:
torch.save(net.state_dict(), 'trained_mlp.pth')

Load the Trained Model

In [11]:
new_net = MLPNet().to(device)

new_net.load_state_dict(torch.load('trained_mlp.pth'))

<All keys matched successfully>

Evaluate on Test Data

In [12]:
correct = 0
total = 0

new_net.eval()

with torch.no_grad():

    for data in test_loader:

        inputs, labels = data

        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = new_net(inputs)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

print(f'Accuracy on test images: {100 * correct / total:.2f}%')

Accuracy on test images: 56.67%


Test with Unseen Images

In [14]:
# Transform for new images
new_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])


def load_image(image_path):

    image = Image.open(image_path).convert('RGB')

    image = new_transform(image)

    # Add batch dimension
    image = image.unsqueeze(0)

    return image


image_path = 'example2.jpg'

image = load_image(image_path).to(device)

new_net.eval()

with torch.no_grad():

    output = new_net(image)

    _, predicted = torch.max(output, 1)

print(f'Predicted class: {class_names[predicted.item()]}')

Predicted class: car
